In [ ]:
from typing import TypedDict, Literal
import sqlite3
import re

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI


# =========================
# LLM 설정
# =========================

llm = ChatOpenAI(
    model="local-model",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0
)


# =========================
# DB 준비
# =========================

conn = sqlite3.connect(":memory:", check_same_thread=False)
cur = conn.cursor()

cur.execute("""
CREATE TABLE sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    branch TEXT,
    product_name TEXT,
    sales_amount INTEGER,
    sales_date TEXT
)
""")

cur.executemany("""
INSERT INTO sales (branch, product_name, sales_amount, sales_date)
VALUES (?, ?, ?, ?)
""", [
    ("서울", "사과", 12000, "2026-04-01"),
    ("서울", "바나나", 8000, "2026-04-02"),
    ("부산", "사과", 15000, "2026-04-03"),
    ("서울", "딸기", 22000, "2026-04-04"),
])

conn.commit()


# =========================
# Text-to-SQL State
# =========================

class TextToSQLState(TypedDict):
    question: str
    schema: str
    sql: str
    result: str
    error: str
    retry_count: int
    answer: str


# =========================
# A-1. DB 스키마 조회
# =========================

def get_schema_node(state: TextToSQLState) -> TextToSQLState:
    cur = conn.cursor()

    cur.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """)

    tables = cur.fetchall()
    schema_text = ""

    for (table_name,) in tables:
        schema_text += f"\nTABLE: {table_name}\n"

        cur.execute(f"PRAGMA table_info({table_name})")
        columns = cur.fetchall()

        for col in columns:
            col_name = col[1]
            col_type = col[2]
            schema_text += f"- {col_name}: {col_type}\n"

    return {
        **state,
        "schema": schema_text
    }


# =========================
# A-2. SQL 생성
# =========================

def generate_sql_node(state: TextToSQLState) -> TextToSQLState:
    prompt = f"""
너는 Text-to-SQL 생성기다.

아래 DB 스키마를 보고 사용자 질문에 맞는 SQLite SELECT SQL만 작성해라.

[DB Schema]
{state["schema"]}

[User Question]
{state["question"]}

규칙:
- SELECT 문만 작성
- INSERT, UPDATE, DELETE, DROP, ALTER 금지
- 설명 없이 SQL만 출력
"""

    response = llm.invoke(prompt)
    sql = response.content.strip()

    sql = sql.replace("```sql", "").replace("```", "").strip()

    return {
        **state,
        "sql": sql,
        "error": ""
    }


# =========================
# A-3. SQL 검증
# =========================

def validate_sql_node(state: TextToSQLState) -> TextToSQLState:
    sql = state["sql"]
    normalized = sql.lower().strip()

    if not normalized.startswith("select"):
        return {
            **state,
            "error": "SELECT 문만 실행할 수 있습니다.",
            "retry_count": state["retry_count"] + 1
        }

    blocked = [
        "insert", "update", "delete", "drop", "alter",
        "truncate", "create", "replace"
    ]

    for keyword in blocked:
        if re.search(rf"\b{keyword}\b", normalized):
            return {
                **state,
                "error": f"위험한 SQL 키워드가 포함되어 있습니다: {keyword}",
                "retry_count": state["retry_count"] + 1
            }

    return {
        **state,
        "error": ""
    }


# =========================
# A-3 이후 분기
# =========================

def route_after_validation(
    state: TextToSQLState
) -> Literal["execute", "retry", "fail"]:
    if state["error"] == "":
        return "execute"

    if state["retry_count"] < 2:
        return "retry"

    return "fail"


# =========================
# A-4. SQL 실행
# =========================

def execute_sql_node(state: TextToSQLState) -> TextToSQLState:
    try:
        cur = conn.cursor()
        cur.execute(state["sql"])

        columns = [desc[0] for desc in cur.description]
        rows = cur.fetchall()

        result = {
            "columns": columns,
            "rows": rows
        }

        return {
            **state,
            "result": str(result),
            "error": ""
        }

    except Exception as e:
        return {
            **state,
            "result": "",
            "error": str(e)
        }


# =========================
# A-5. 결과 요약
# =========================

def summarize_node(state: TextToSQLState) -> TextToSQLState:
    if state["error"]:
        answer = f"SQL 처리 중 오류가 발생했습니다: {state['error']}"
    else:
        prompt = f"""
사용자 질문:
{state["question"]}

실행한 SQL:
{state["sql"]}

조회 결과:
{state["result"]}

위 내용을 한국어로 짧게 요약해줘.
"""
        response = llm.invoke(prompt)
        answer = response.content.strip()

    return {
        **state,
        "answer": answer
    }


# =========================
# Text-to-SQL Subgraph 생성
# =========================

text_to_sql_builder = StateGraph(TextToSQLState)

text_to_sql_builder.add_node("get_schema", get_schema_node)
text_to_sql_builder.add_node("generate_sql", generate_sql_node)
text_to_sql_builder.add_node("validate_sql", validate_sql_node)
text_to_sql_builder.add_node("execute_sql", execute_sql_node)
text_to_sql_builder.add_node("summarize", summarize_node)

text_to_sql_builder.add_edge(START, "get_schema")
text_to_sql_builder.add_edge("get_schema", "generate_sql")
text_to_sql_builder.add_edge("generate_sql", "validate_sql")

text_to_sql_builder.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "execute": "execute_sql",
        "retry": "generate_sql",
        "fail": "summarize"
    }
)

text_to_sql_builder.add_edge("execute_sql", "summarize")
text_to_sql_builder.add_edge("summarize", END)

text_to_sql_graph = text_to_sql_builder.compile()